<a href="https://colab.research.google.com/github/aryasure/BA305-UK-Mapping-Musem-Project/blob/main/Museums_Cleaned_Datset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score
)

import matplotlib.pyplot as plt
import seaborn as sns

In [31]:
df = pd.read_csv('museums.csv')
df

,museum_id,Name_of_museum,Address_line_1,Address_line_2,"Village,_Town_or_City",Postcode,Latitude,Longitude,Admin_area,Accreditation,Governance,Size,Size_provenance,Subject_Matter,Year_opened,Year_closed,DOMUS_Subject_Matter,DOMUS_identifier,Primary_provenance_of_data,Identifier_used_in_primary_data_source,Area_Deprivation_index,Area_Deprivation_index_crime,Area_Deprivation_index_education,Area_Deprivation_index_employment,Area_Deprivation_index_health,Area_Deprivation_index_housing,Area_Deprivation_index_income,Area_Deprivation_index_services,Area_Geodemographic_group,Area_Geodemographic_group_code,Area_Geodemographic_subgroup,Area_Geodemographic_subgroup_code,Area_Geodemographic_supergroup,Area_Geodemographic_supergroup_code,Notes
0,mm.New.1,Titanic Belfast,1 Olympic Way,NaN,Belfast,BT3 9EP,54.608080,-5.909915,/Northern Ireland/Belfast (NI Loc Gov District),Unaccredited,Independent-Not_for_profit,large,NaN,Sea_and_seafaring-Boats_and_ships,2012:2012,9999:9999,NaN,NaN,New,NaN,2.0,3.0,1.0,2.0,1.0,4.0,5.0,5.0,Larger Towns and Cities,2ar,Larger Towns and Cities,2a1r,Business Education and Heritage Centres,2r,NaN
1,mm.aim.1230,The Woodland Heritage Museum,NaN,Brokerswood,nr Westbury,BA13 4EH,51.270903,-2.231863,/England/South West (English Region)/Wiltshire...,Unaccredited,Independent-Unknown,small,aim_size_designation,Natural_world-Other,1971:1971,2007:2017,NaN,NaN,aim,NaN,9.0,8.0,9.0,8.0,8.0,6.0,8.0,4.0,Country Living,7ar,Country Living,7a1r,Town and Country Living,7r,Previously known as Phillips Countryside Museu...
2,mm.domus.WM019,Warwickshire Museum Of Rural Life,Warwick College of Agriculture,Horticulture & Equine Studies,Moreton Morrell,CV35 9BL,52.197146,-1.555528,/England/West Midlands (English Region)/Warwic...,Unaccredited,Government-Local_Authority,medium,mm_prediction_random_forest,Rural_Industry-Farming,1984:1984,9999:9999,NaN,1218.0,domus,WM000019,8.0,9.0,8.0,10.0,8.0,5.0,8.0,1.0,English and Welsh Countryside,3ar,Older Farming Communities,3a1r,Countryside Living,3r,NaN
3,mm.aim.0484,Battle Of Flowers Museum,La Robeline,Mont des Corvees,St Ouen,JE3 2ES,49.220770,-2.107120,/Channel Islands,Unaccredited,Independent-Private,small,aim_size_designation,Arts-Crafts,1971:1971,2012:2012,NaN,NaN,aim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mm.aim.0485,Jet Age Museum,57 Crown Drive,Bishops Cleeve,Cheltenham,GL52 8TA,51.940092,-2.060766,/England/South West (English Region)/Glouceste...,Accredited,Independent-Not_for_profit,medium,aim_size_designation,War_and_conflict-Airforce,2013:2013,9999:9999,NaN,NaN,aim,NaN,8.0,10.0,7.0,7.0,8.0,7.0,8.0,4.0,Country Living,7ar,Country Living,7a1r,Town and Country Living,7r,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4186,mm.domus.WA018,y Gaer,Glamorgan Street,NaN,Brecon,LD3 7DW,51.946352,-3.390436,/Wales/Powys (Welsh UA),Accredited,Government-Local_Authority,medium,domus,Local_Histories,1928:1928,9999:9999,social history,1134.0,domus,WA000018,7.0,1.0,8.0,7.0,8.0,2.0,7.0,8.0,English and Welsh Countryside,3ar,Sparse English and Welsh Countryside,3a2r,Countryside Living,3r,Museum was revamped and renamed in 2019 to inc...
4187,mm.domus.SE332,Jewish Museum - Finchley,The Sternberg Centre,80 East End Road,London,N3 2SY,51.595298,-0.189303,/England/London (English Region)/Barnet (Londo...,Unaccredited,Independent-Not_for_profit,medium,domus,Belief_and_identity-Religion,1983:1983,2007:2007,social history,340.0,domus,SE000332,6.0,5.0,9.0,6.0,9.0,5.0,5.0,2.0,Ethnically Diverse Metropolitan Living,4ar,Ethnically Diverse Metropolitan Living,4a1r,Ethnically Diverse Metropolitan Living,4r,NaN
4188,mm.aim.0067,Bawdsey Radar,Transmitter Block,NaN,Bawdsey,IP12 3BA,51.991166,1.403723,/England/East of England (English Region)/Suff...,Unaccredited,Independent-Not_for_profit,small,visitbritain,War_and_conflict-Other,2003:2003,9999:9999,NaN,NaN,aim,NaN,6.0,8.0,6.0,8.0,9.0,1.0,8.0,1.0,

In [32]:
# Missing values by column
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
}).sort_values("missing_pct", ascending=False)

print(missing)

# Check exact duplicate rows and duplicate IDs
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate museum IDs:", df["museum_id"].duplicated().sum())
print("Duplicate museum names:", df["Name_of_museum"].duplicated().sum())

                                        missing_count  ...  unique_values
Notes                                            2980  ...            956
Address_line_2                                   2816  ...           1167
DOMUS_Subject_Matter                             2788  ...             21
DOMUS_identifier                                 2347  ...           1843
Identifier_used_in_primary_data_source           2056  ...           2134
Address_line_1                                    441  ...           3212
Size_provenance                                   179  ...             16
Area_Geodemographic_subgroup                       49  ...             25
Area_Geodemographic_subgroup_code                  49  ...             24
Area_Geodemographic_group                          49  ...             17
Area_Geodemographic_group_code                     49  ...             16
Area_Deprivation_index_education                   49  ...             10
Area_Deprivation_index_housing        

In [33]:
#Drop unnecessary columns
columns_to_drop = [
    "DOMUS_Subject_Matter",
    "DOMUS_identifier",
    "Primary_provenance_of_data",
    "Identifier_used_in_primary_data_source",
    "Notes",
    "Size_provenance",
    "Address_line_2"
]

df = df.drop(columns=columns_to_drop)

In [34]:
# Convert column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

# Remove leading/trailing spaces in text columns
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    df[col] = df[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Standardize blank strings as missing values
df[text_cols] = df[text_cols].replace(r"^\s*$", np.nan, regex=True)

In [35]:
#Clean museum size using the mode
df["size"] = df["size"].replace("unknown", pd.NA)
most_common_size = df["size"].mode()[0]
df["size"] = df["size"].fillna(most_common_size)

In [36]:
#Clean years columns
df["operating_status"] = np.where(
    df["year_closed"] == "9999:9999",
    "open",
    "closed"
)

# Drop the original year_closed column
df = df.drop(columns=["year_closed"])

In [37]:
#Clean subject matter
df["subject_matter"] = (
    df["subject_matter"]
    .astype("string")
    .str.strip()
    .fillna("Unknown")
)

In [38]:
#Remove invalid coordinates
df = df[
    df["latitude"].between(-90, 90) &
    df["longitude"].between(-180, 180)
].copy()


In [41]:
print(df.shape)
print(df.isnull().sum())
print(df["size"].value_counts())

(4190, 29)
museum_id                                 0
name_of_museum                            0
address_line_1                          441
address_line_2                         2815
village_town_or_city                      4
postcode                                  0
latitude                                  0
longitude                                 0
admin_area                                0
accreditation                             0
governance                                0
size                                      0
subject_matter                            0
year_opened                               0
area_deprivation_index                   48
area_deprivation_index_crime             48
area_deprivation_index_education         48
area_deprivation_index_employment        48
area_deprivation_index_health            48
area_deprivation_index_housing           48
area_deprivation_index_income            48
area_deprivation_index_services          48
area_geodemographic_g

In [42]:
# Save a cleaned copy
df.to_csv("museums_cleaned.csv", index=False)

print("Cleaned dataset shape:", df.shape)
print("Saved museums_cleaned.csv")

Cleaned dataset shape: (4190, 29)
Saved museums_cleaned.csv
